In [1]:
# =============================================================================
# 03_silver: Clean, type and validate each Bronze table
# -----------------------------------------------------------------------------
# Input : output/bronze/*.csv   (append-only, everything is text)
# Output: output/silver/*.csv   (one CSV per source table, current state only)
#
# Steps for every table:
#   1. Read Bronze, trim text, turn empty strings into NULL
#   2. Remove exact duplicate rows
#   3. Upsert on the natural key: keep only the latest version (_ingested_at)
#   4. Cast columns to proper types (dates, decimals, integers)
#   5. Flag bad records in _is_valid, with the reason in _invalid_reason
#      (bad records are KEPT and flagged, never dropped silently)
#
# NULL HANDLING STRATEGY (per field)
#   Keys / IDs ................ never filled; a missing key makes the record invalid
#   order dates (approved, carrier, delivered)
#                              NULL is legitimate (order not shipped/delivered yet), kept as NULL
#   order_purchase_timestamp .. required; NULL makes the record invalid
#   price, freight_value ...... required; NULL makes the record invalid (we never invent money)
#   payment_value ............. required; NULL makes the record invalid
#   product_category_name ..... filled with "unknown" (as required by the Gold spec)
#   product weight/dimensions . kept NULL (0 would be a fake value and would distort volume)
#   city / state / zip ........ kept NULL when missing (not needed to identify a record)
#
# Idempotency: Silver is rebuilt in full from Bronze on every run and written
# with overwrite. The same Bronze always gives the same Silver, so re-running
# can never create duplicates.
# =============================================================================

import os, glob
import pandas as pd
from pyspark.sql import SparkSession, Window, Column, functions as F
from pyspark.sql.types import DecimalType

# ---- Spark session -----------------------------------------------------------
# Session time zone is fixed to UTC so timestamps behave the same on every machine.
# shuffle.partitions is lowered because our data is small (default 200 is slow locally).
spark = (SparkSession.builder
         .master("local[*]")
         .appName("blue-owls-silver")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

# ---- Paths and formats -------------------------------------------------------
OUTPUT_DIR = "/home/jovyan/work/output"
BRONZE_DIR = f"{OUTPUT_DIR}/bronze"
SILVER_DIR = f"{OUTPUT_DIR}/silver"
os.makedirs(SILVER_DIR, exist_ok=True)

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"   # timestamp format used by the API
MONEY = DecimalType(12, 2)          # price, freight, payments: exact decimals, not floats

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
# Collects row counts after each cleaning step, so no record disappears silently.
dq_log = []


def read_bronze(endpoint):
    """Read one Bronze CSV with every column as text (no schema guessing,
    so zip codes like '01046' keep their leading zero)."""
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)
            .option("escape", '"')
            .csv(f"{BRONZE_DIR}/{endpoint}.csv"))


def trim_all_strings(df):
    """Trim spaces and convert empty strings to NULL in every column."""
    for c in df.columns:
        trimmed = F.trim(F.col(c))
        df = df.withColumn(c, F.when(trimmed == "", F.lit(None)).otherwise(trimmed))
    return df


def keep_latest(df, partition_cols):
    """Within each group of partition_cols keep only the most recently ingested row."""
    w = Window.partitionBy(*partition_cols).orderBy(F.col("_ingested_at").desc())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))


def deduplicate(df, key_cols, table):
    """
    Step 1: remove exact duplicate rows (all business columns identical).
    Step 2: upsert on the natural key: if the same key was ingested more than once
            (e.g. a record changed between runs), only the latest version survives.
    (Key columns are never NULL here: Part 1 quarantines records with empty keys.)
    """
    business_cols = [c for c in df.columns if not c.startswith("_")]
    n_bronze = df.count()
    df = keep_latest(df, business_cols)      # exact duplicates
    n_after_exact = df.count()
    df = keep_latest(df, key_cols)           # upsert by natural key
    n_final = df.count()
    dq_log.append({"table": table,
                   "bronze_rows": n_bronze,
                   "exact_duplicates_removed": n_bronze - n_after_exact,
                   "older_versions_replaced": n_after_exact - n_final,
                   "silver_rows": n_final})
    return df


def apply_casts(df, casts):
    """
    casts = {column_name: function(Column) -> typed Column}
    Converts each column to its proper type. If the text was present but the
    conversion produced NULL (a bad value like 'abc'), the column name is recorded
    in _cast_issues so the record can be flagged instead of silently losing its value.
    """
    issue_flags = []
    for name, fn in casts.items():
        issue_flags.append(F.when(F.col(name).isNotNull() & fn(F.col(name)).isNull(), F.lit(name)))
    df = df.withColumn("_cast_issues", F.concat_ws(",", *issue_flags))   # "" when no issues
    for name, fn in casts.items():
        df = df.withColumn(name, fn(F.col(name)))
    return df


def flag_validity(df, rules):
    """
    rules = list of (reason, failure_condition).
    A record is invalid if ANY condition is true. _invalid_reason lists all reasons
    joined by '; ' (NULL when the record is valid), _is_valid is the boolean flag.
    """
    reasons = [F.when(cond, reason if isinstance(reason, Column) else F.lit(reason))
               for reason, cond in rules]
    df = df.withColumn("_invalid_reason", F.concat_ws("; ", *reasons))
    df = df.withColumn("_invalid_reason",
                       F.when(F.col("_invalid_reason") == "", F.lit(None)).otherwise(F.col("_invalid_reason")))
    return df.withColumn("_is_valid", F.col("_invalid_reason").isNull())


# Reusable rule: report columns whose text could not be converted to its type
CAST_RULE = (F.concat(F.lit("unparseable values: "), F.col("_cast_issues")),
             F.col("_cast_issues") != "")


def write_single_csv(df, name):
    """
    Spark normally writes a folder of part files. The assignment wants ONE CSV per table,
    so we write a single partition to a temp folder, then move the part file to <name>.csv.
    mode=overwrite makes re-runs safe.
    """
    tmp_dir = f"{SILVER_DIR}/_tmp_{name}"
    final_path = f"{SILVER_DIR}/{name}.csv"
    (df.coalesce(1).write.mode("overwrite")
       .option("header", True)
       .option("timestampFormat", TS_FORMAT)
       .csv(tmp_dir))
    part_file = glob.glob(f"{tmp_dir}/part-*.csv")[0]
    os.replace(part_file, final_path)
    import shutil
    shutil.rmtree(tmp_dir)

print("Helpers ready")

Helpers ready


In [3]:
def build_customers():
    df = trim_all_strings(read_bronze("customers"))
    df = deduplicate(df, ["customer_id"], "customers")
    df = (df
          # Zip prefixes are always 5 digits; pad so '1046' and '01046' can never differ
          .withColumn("customer_zip_code_prefix", F.lpad(F.col("customer_zip_code_prefix"), 5, "0"))
          .withColumn("customer_city", F.lower(F.col("customer_city")))    # consistent casing
          .withColumn("customer_state", F.upper(F.col("customer_state"))))
    df = flag_validity(df, [
        ("customer_id is missing", F.col("customer_id").isNull()),
        ("customer_unique_id is missing", F.col("customer_unique_id").isNull()),
    ])
    return df.drop("_source_endpoint")


def build_sellers():
    df = trim_all_strings(read_bronze("sellers"))
    df = deduplicate(df, ["seller_id"], "sellers")
    df = (df
          .withColumn("seller_zip_code_prefix", F.lpad(F.col("seller_zip_code_prefix"), 5, "0"))
          .withColumn("seller_city", F.lower(F.col("seller_city")))
          .withColumn("seller_state", F.upper(F.col("seller_state"))))
    df = flag_validity(df, [
        ("seller_id is missing", F.col("seller_id").isNull()),
    ])
    return df.drop("_source_endpoint")


def build_products():
    df = trim_all_strings(read_bronze("products"))
    df = deduplicate(df, ["product_id"], "products")

    # The source dataset misspells "length" as "lenght"; fix the names from Silver onward.
    df = (df.withColumnRenamed("product_name_lenght", "product_name_length")
            .withColumnRenamed("product_description_lenght", "product_description_length"))

    df = apply_casts(df, {
        "product_name_length":        lambda c: c.cast("int"),
        "product_description_length": lambda c: c.cast("int"),
        "product_photos_qty":         lambda c: c.cast("int"),
        "product_weight_g":           lambda c: c.cast(MONEY),
        "product_length_cm":          lambda c: c.cast(MONEY),
        "product_height_cm":          lambda c: c.cast(MONEY),
        "product_width_cm":           lambda c: c.cast(MONEY),
    })

    # Missing category becomes "unknown" (required by the Gold spec)
    df = df.withColumn("product_category_name",
                       F.coalesce(F.lower(F.col("product_category_name")), F.lit("unknown")))

    df = flag_validity(df, [
        ("product_id is missing", F.col("product_id").isNull()),
        ("negative weight", F.col("product_weight_g") < 0),
        ("negative dimension", (F.col("product_length_cm") < 0) | (F.col("product_height_cm") < 0)
                               | (F.col("product_width_cm") < 0)),
        CAST_RULE,
    ])
    return df.drop("_source_endpoint", "_cast_issues")

print("Customers / sellers / products ready")

Customers / sellers / products ready


In [4]:
def to_ts(c):
    """Convert text to a timestamp; invalid text becomes NULL (then flagged by apply_casts)."""
    return F.to_timestamp(c, TS_FORMAT)


def build_orders():
    df = trim_all_strings(read_bronze("orders"))
    df = deduplicate(df, ["order_id"], "orders")

    df = apply_casts(df, {
        "order_purchase_timestamp":      to_ts,
        "order_approved_at":             to_ts,
        "order_delivered_carrier_date":  to_ts,
        "order_delivered_customer_date": to_ts,
        "order_estimated_delivery_date": to_ts,
    })
    df = df.withColumn("order_status", F.lower(F.col("order_status")))   # consistent casing

    df = flag_validity(df, [
        ("order_id is missing", F.col("order_id").isNull()),
        ("customer_id is missing", F.col("customer_id").isNull()),
        ("purchase timestamp is missing", F.col("order_purchase_timestamp").isNull()),
        # Impossible: the order cannot arrive before it was placed
        ("delivered before purchase",
         F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp")),
        CAST_RULE,
    ])
    return df.drop("_source_endpoint", "_cast_issues")


def build_order_items(orders_df):
    df = trim_all_strings(read_bronze("order_items"))
    df = deduplicate(df, ["order_id", "order_item_id"], "order_items")

    df = apply_casts(df, {
        "order_item_id":       lambda c: c.cast("int"),
        "shipping_limit_date": to_ts,
        "price":               lambda c: c.cast(MONEY),
        "freight_value":       lambda c: c.cast(MONEY),
    })

    # Orphan check: does this item's order_id exist in Silver orders?
    known_orders = orders_df.select("order_id").distinct().withColumn("_order_found", F.lit(True))
    df = df.join(known_orders, on="order_id", how="left")

    df = flag_validity(df, [
        ("order_id is missing", F.col("order_id").isNull()),
        ("product_id is missing", F.col("product_id").isNull()),
        ("seller_id is missing", F.col("seller_id").isNull()),
        ("price is missing", F.col("price").isNull()),
        ("price is negative", F.col("price") < 0),
        ("freight_value is missing", F.col("freight_value").isNull()),
        ("freight_value is negative", F.col("freight_value") < 0),
        ("no matching order_id in orders", F.col("_order_found").isNull()),
        CAST_RULE,
    ])
    return df.drop("_source_endpoint", "_cast_issues", "_order_found")

print("Orders / order_items ready")

Orders / order_items ready


In [5]:
def build_payments(orders_df):
    df = trim_all_strings(read_bronze("payments"))
    df = deduplicate(df, ["order_id", "payment_sequential"], "payments")

    df = apply_casts(df, {
        "payment_sequential":   lambda c: c.cast("int"),
        "payment_installments": lambda c: c.cast("int"),
        "payment_value":        lambda c: c.cast(MONEY),
    })
    df = df.withColumn("payment_type", F.lower(F.col("payment_type")))

    # Same orphan check as order_items. NOTE: the API cannot date-filter payments, so
    # payments of orders before 2018-07-01 arrive too. They have no matching order in
    # our scoped orders table and are flagged here; Gold only uses valid rows.
    known_orders = orders_df.select("order_id").distinct().withColumn("_order_found", F.lit(True))
    df = df.join(known_orders, on="order_id", how="left")

    df = flag_validity(df, [
        ("order_id is missing", F.col("order_id").isNull()),
        ("payment_value is missing", F.col("payment_value").isNull()),
        ("payment_value is negative", F.col("payment_value") < 0),
        ("negative installments", F.col("payment_installments") < 0),
        ("no matching order (outside 2018-07-01 scope or missing)", F.col("_order_found").isNull()),
        CAST_RULE,
    ])
    return df.drop("_source_endpoint", "_cast_issues", "_order_found")

print("Payments ready")

Payments ready


In [6]:
dq_log.clear()   # fresh log if this cell is re-run

silver = {}
silver["customers"]   = build_customers()
silver["sellers"]     = build_sellers()
silver["products"]    = build_products()
silver["orders"]      = build_orders().cache()   # cached: reused by order_items and payments
silver["order_items"] = build_order_items(silver["orders"])
silver["payments"]    = build_payments(silver["orders"])

for name, df in silver.items():
    write_single_csv(df, name)
    print(f"written: {SILVER_DIR}/{name}.csv")

written: /home/jovyan/work/output/silver/customers.csv
written: /home/jovyan/work/output/silver/sellers.csv
written: /home/jovyan/work/output/silver/products.csv
written: /home/jovyan/work/output/silver/orders.csv
written: /home/jovyan/work/output/silver/order_items.csv
written: /home/jovyan/work/output/silver/payments.csv


In [7]:
# 1) What happened to the rows at each step (nothing disappears without a count)
print("===== DEDUPLICATION =====")
print(pd.DataFrame(dq_log).to_string(index=False))

# 2) Valid vs invalid records per table
rows = []
for name, df in silver.items():
    total = df.count()
    valid = df.filter(F.col("_is_valid")).count()
    rows.append({"table": name, "rows": total, "valid": valid, "invalid": total - valid})
print("\n===== VALIDITY =====")
print(pd.DataFrame(rows).to_string(index=False))

# 3) Why records are invalid
for name, df in silver.items():
    bad = df.filter(~F.col("_is_valid"))
    if bad.count() > 0:
        print(f"\n{name}: invalid records by reason")
        bad.groupBy("_invalid_reason").count().orderBy(F.desc("count")).show(truncate=False)

# 4) Scope check: orders must start on/after 2018-07-01
print("Orders date range:")
silver["orders"].agg(F.min("order_purchase_timestamp").alias("first_order"),
                     F.max("order_purchase_timestamp").alias("last_order")).show(truncate=False)

===== DEDUPLICATION =====
      table  bronze_rows  exact_duplicates_removed  older_versions_replaced  silver_rows
  customers        99441                         0                        0        99441
    sellers         3095                         0                        0         3095
   products        32951                         0                        0        32951
     orders        12824                         0                        0        12824
order_items        15590                         0                        0        15590
   payments       103886                         0                        0       103886

===== VALIDITY =====
      table   rows  valid  invalid
  customers  99441  99441        0
    sellers   3095   3095        0
   products  32951  32951        0
     orders  12824  12824        0
order_items  15590  14341     1249
   payments 103886  13225    90661

order_items: invalid records by reason
+------------------------------+-----+
|_inv

In [8]:
# Natural key of each table: must be unique in Silver (that is what "upsert" guarantees)
KEYS = {
    "customers":   ["customer_id"],
    "sellers":     ["seller_id"],
    "products":    ["product_id"],
    "orders":      ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments":    ["order_id", "payment_sequential"],
}

print(f"{'table':12s} {'rows':>8s} {'unique_keys':>12s} {'csv_rows':>9s}  result")
for name, keys in KEYS.items():
    df = silver[name]
    total = df.count()
    distinct = df.select(*keys).distinct().count()
    # Read the CSV back from disk to prove the file really contains everything
    csv_rows = spark.read.option("header", True).csv(f"{SILVER_DIR}/{name}.csv").count()
    ok = (total == distinct == csv_rows)
    print(f"{name:12s} {total:>8d} {distinct:>12d} {csv_rows:>9d}  {'OK' if ok else 'MISMATCH'}")

print("\nSample of silver orders:")
spark.read.option("header", True).csv(f"{SILVER_DIR}/orders.csv").show(3, truncate=False)

table            rows  unique_keys  csv_rows  result
customers       99441        99441     99441  OK
sellers          3095         3095      3095  OK
products        32951        32951     32951  OK
orders          12824        12824     12824  OK
order_items     15590        15590     15590  OK
payments       103886       103886    103886  OK

Sample of silver orders:
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------------+---------------+---------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_ingested_at                    |_invalid_reason|_is_valid|
+--------------------------------+--------------------------------+------------